In [1]:
import pandas as pd
import numpy as np

LOAD DATASET

In [2]:
df=pd.read_excel("/content/sales data.xlsx")
print("Original Dataset:")
print(df.head())
print("\nDataset Shape:",df.shape)

Original Dataset:
    country  order_value_EUR       cost                 date     category  \
0    Sweden         98320.37   77722.25            8/23/2020        Games   
1    France         46296.26   40319.41            5/15/2020        Games   
2  Portugal        140337.34  115708.14  2020-04-09 00:00:00   Appliances   
3    France        203604.46  175344.16            6/26/2019  Electronics   
4        UK         63979.04   56032.84           10/22/2019        Games   

     customer_name    sales_manager            sales_rep device_type  \
0   Konopelski LLC     Maxie Marrow   Tarrah Castelletti      Tablet   
1       Wisoky Inc    Othello Bowes  Amelina Piscopiello      Tablet   
2    Hegmann Group  Celine Tumasian        Corene Shirer          PC   
3  Kirlin and Sons    Othello Bowes         Crysta Halls      Mobile   
4   Schoen-Keeling  Jessamine Apark   Genevra Charrisson          PC   

     order_id  
0  70-0511466  
1  77-3489084  
2  65-8218141  
3  29-5478106  
4  27-

CHECK MISSING VALUES

In [3]:
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
country            0
order_value_EUR    5
cost               0
date               0
category           0
customer_name      0
sales_manager      0
sales_rep          0
device_type        5
order_id           0
dtype: int64


CHECK DUPLICATES

In [5]:
print("\nNumber of Duplicate Rows:")
print(df.duplicated().sum())
df=df.drop_duplicates()
print("\nShape after removing duplicates:")
print(df.shape)


Number of Duplicate Rows:
1

Shape after removing duplicates:
(1000, 10)


STANDARDIZE COLUMN NAMES

In [7]:
df.columns=df.columns.str.strip().str.lower().str.replace(" ","_")
print("\nColumn Names:")
print(df.columns)


Column Names:
Index(['country', 'order_value_eur', 'cost', 'date', 'category',
       'customer_name', 'sales_manager', 'sales_rep', 'device_type',
       'order_id'],
      dtype='object')


CLEAN TEXT COLUMNS

In [8]:
text_columns=[
    "country",
    "category",
    "customer_name",
    "sales_manager",
    "sales_rep",
    "device_type",
    "order_id"
]
for col in text_columns:
  if col in df.columns:
    df[col]=df[col].astype("string").str.strip()

STANDARDIZE CATEGORICAL VALUES

In [12]:
if "country" in df.columns:
  df["country"]=df["country"].str.title()
  df["country"]=df["country"].replace({
      "uk":"UK",
      "u.k.":"UK",
      "u.k.":"UK"
  })
  if "category" in df.columns:
    df["category"]=df["category"].str.title()
  if "device_type" in df.columns:
    df["device_type"]=df["device_type"].str.title()

CONVERT NUMERIC COLUMNS

In [13]:
df["order_value_eur"]=pd.to_numeric(

df["order_value_eur"],
     errors="coerce"
     )
df["cost"]=pd.to_numeric(
    df["cost"],
    errors="coerce"
)

CONVERT DATA COLUMN

In [14]:
df["date"]=pd.to_datetime(
    df["date"],
    errors="coerce"
)
df["date"]=df["date"].dt.normalize()

HANDLE MISSING ORDER VALUE

In [15]:
df["cost_order_ratio"]=(
    df["cost"]/df["order_value_eur"]
)
group_ratio=df.groupby(
    ["country","category"]

)["cost_order_ratio"].transform("median")

missing_order=df["order_value_eur"].isnull()
df.loc[missing_order,"order_value_eur"]=(
    df.loc[missing_order,"cost"]/
    group_ratio[missing_order]
)



HANDLE INVALID/MISSING COST

In [16]:
df["cost_order_ratio"]=(
    df["cost"]/df["order_value_eur"]
)
missing_cost=df["cost"].isnull()
df.loc[missing_cost,"cost"]=(
    df.loc[missing_cost,"order_value_eur"] *
    group_ratio[missing_cost]
)

HANDLE MISSING DEVICE TYPE

In [19]:
device_mode=df["device_type"].mode()[0]
df["device_type"]=df["device_type"].fillna(
    device_mode
)

OUTLIER DETECTION USING IQR

In [22]:
def detect_outliers(column):
  Q1=df[column].quantile(0.25)
  Q3=df[column].quantile(0.75)
  IQR=Q3-Q1
  lower_limit=Q1 - 1.5 * IQR
  upper_limit=Q3 + 1.5 * IQR
  outliers = (
      (df[column]< lower_limit) |
      (df[column]> upper_limit)
  )
  return outliers,lower_limit,upper_limit

df["order_value_outlier"],lower_ov,upper_ov=(
    detect_outliers("order_value_eur")
)
df["cost_outlier"],lower_cost,upper_cost=(
    detect_outliers("cost")
)
df["ratio_outlier"],lower_ratio,upper_ratio=(
    detect_outliers("cost_order_ratio")
)

COMBINE OUTLIER FLAGS

In [24]:
df["outlier_flag"]=(
    df["order_value_outlier"] |
    df["cost_outlier"] |
    df["ratio_outlier"]
)

DISPLAY OUTLIERS

In [27]:
outliers=df[df["outlier_flag"]]
print("\nNumber of Outlier Rows: ")
print(len(outliers))
print("\nOutlier Records: ")
print("outliers")


Number of Outlier Rows: 
18

Outlier Records: 
outliers


BUSINESS RULE VALIDATION

In [28]:
df["cost_exceeds_order_value"]=(
    df["cost"]>df["order_value_eur"]

)
print("\nRecords where Cost > Order Value: ")
print(
    df[df["cost_exceeds_order_value"]]
)


Records where Cost > Order Value: 
Empty DataFrame
Columns: [country, order_value_eur, cost, date, category, customer_name, sales_manager, sales_rep, device_type, order_id, cost_order_ratio, order_value_outlier, cost_outlier, ratio_outlier, outlier_flag, cost_exceeds_order_value]
Index: []


CALCULATE PROFIT / MARGIN

In [29]:
df["gross_margin_eur"]=(
    df["order_value_eur"] - df["cost"]

)
df["gross_margin_pct"]=(
    df["gross_margin_eur"] /
    df["order_value_eur"]

) * 100

FINAL MISSING VALUE CHECK

In [30]:
print("\nRemaining Missing Values: ")
print(df.isnull().sum())


Remaining Missing Values: 
country                     0
order_value_eur             1
cost                        0
date                        0
category                    0
customer_name               0
sales_manager               0
sales_rep                   0
device_type                 0
order_id                    0
cost_order_ratio            2
order_value_outlier         0
cost_outlier                0
ratio_outlier               0
outlier_flag                0
cost_exceeds_order_value    0
gross_margin_eur            1
gross_margin_pct            1
dtype: int64


FINAL DATASET INFORMATION

In [31]:
print("\nFinal Datset Shape: ")
print(df.shape)

print("\nFinal Dataset: ")
print(df.head())


Final Datset Shape: 
(1000, 18)

Final Dataset: 
    country  order_value_eur       cost       date     category  \
0    Sweden         98320.37   77722.25 2020-08-23        Games   
1    France         46296.26   40319.41 2020-05-15        Games   
2  Portugal        140337.34  115708.14 2020-04-09   Appliances   
3    France        203604.46  175344.16 2019-06-26  Electronics   
4        Uk         63979.04   56032.84 2019-10-22        Games   

     customer_name    sales_manager            sales_rep device_type  \
0   Konopelski LLC     Maxie Marrow   Tarrah Castelletti      Tablet   
1       Wisoky Inc    Othello Bowes  Amelina Piscopiello      Tablet   
2    Hegmann Group  Celine Tumasian        Corene Shirer          Pc   
3  Kirlin and Sons    Othello Bowes         Crysta Halls      Mobile   
4   Schoen-Keeling  Jessamine Apark   Genevra Charrisson          Pc   

     order_id  cost_order_ratio  order_value_outlier  cost_outlier  \
0  70-0511466            0.7905             

SAVE CLEANED DATA

In [32]:
df.to_excel(
    "sales_data_cleaned.xlsx",
    index=False
)
print("\nCleaned dataset saved successfully!")


Cleaned dataset saved successfully!
